# 03 — ENASIC 2022 (microdatos): ¿por qué no se usan servicios de cuidado?

**Para qué decisión existe este notebook.** El programa candidato es una app que conecte "Necesito cuidado" con "Puedo cuidar". Solo se justifica si la barrera principal es de **conexión**: no saber que existen servicios, o desconfiar de desconocidos. Si la barrera es el **precio**, se necesita un subsidio. Si es la **preferencia por el cuidado familiar**, ningún servicio la resuelve.
La ENASIC es la única fuente que tenemos que pregunta por eso directamente (P6.42–P6.48).

**Alcance.** Nacional. La ENASIC **no** permite estimaciones para la CDMX ni por alcaldía. Nada de lo que sale de aquí se puede presentar como dato de Iztapalapa.

**Método.** Estimaciones ponderadas con el factor de expansión de cada tabla. Varianza por linealización de Taylor con estratos `EST_DIS` y conglomerados `UPM_DIS`: es el mismo diseño que usa el código en R del INEGI. Se aplican los criterios de precisión del INEGI según el coeficiente de variación:
- alta: CV < 15%
- moderada: 15% ≤ CV < 30%
- baja: CV ≥ 30%. Las estimaciones de precisión baja no se usan como hallazgo.

**Universos.** Una misma persona puede estar en varias tablas, así que las tablas **no se suman entre sí**:
- `TPOB_CUI`: personas cuidadoras de 15 años y más. Factor `FAC_CUI`.
- `TPER_ELE`: persona elegida de 15 a 60 años. Factor `FAC_ELE`.
- `THOGAR`: hogares. Factor `FAC_HOG`.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 80)

BASE = Path('datos/enasic')
OUT = Path('resultados'); OUT.mkdir(exist_ok=True)

def tabla(nombre, archivo=None):
    archivo = archivo or f'conjunto_de_datos_{nombre}_enasic_2022.csv'
    df = pd.read_csv(BASE / f'conjunto_de_datos_{nombre}_enasic_2022' / 'conjunto_de_datos' / archivo, dtype=str)
    return df

cui = tabla('tpob_cui'); ele = tabla('tper_ele')
hog = tabla('thogar', 'conjunto_de_datos_thogar_ensasic_2022.csv')   # así viene el nombre (typo del INEGI)
sde = tabla('tcsdempo')
for n, d in [('TPOB_CUI', cui), ('TPER_ELE', ele), ('THOGAR', hog), ('TCSDEMPO', sde)]:
    print(f'{n:9s} registros={len(d):6d}  estratos={d.EST_DIS.nunique()}  UPM={d.UPM_DIS.nunique()}')

TPOB_CUI  registros=  5677  estratos=148  UPM=861
TPER_ELE  registros=  5579  estratos=148  UPM=895
THOGAR    registros=  6508  estratos=148  UPM=896
TCSDEMPO  registros= 21776  estratos=148  UPM=896


## 1. Función de estimación con diseño muestral

Estima una **proporción** (un cociente de totales ponderados) junto con su error estándar, su coeficiente de variación y su intervalo de confianza al 90%, que es el nivel que usa el INEGI.
Los estratos con una sola UPM no aportan a la varianza. El número de estratos en esa situación se reporta.

In [2]:
Z90 = 1.6449

def estimar(df, y, x, peso, etiqueta=''):
    """Proporción ponderada R = Σw·y / Σw·x con varianza por linealización de Taylor.
    y, x: arrays 0/1 (numerador y dominio). Solo cuentan los registros con x = 1."""
    w = pd.to_numeric(df[peso]).to_numpy(float)
    y = np.asarray(y, float); x = np.asarray(x, float)
    X = (w * x).sum(); Y = (w * y).sum(); R = Y / X
    z = w * (y - R * x) / X
    t = pd.DataFrame({'h': df.EST_DIS.values, 'u': df.UPM_DIS.values, 'z': z}).groupby(['h', 'u']).z.sum().reset_index()
    g = t.groupby('h').z
    n_h = g.transform('size'); zbar = g.transform('mean')
    var = ((n_h / (n_h - 1)).where(n_h > 1, 0) * (t.z - zbar) ** 2).sum()
    se = np.sqrt(var); cv = se / R * 100 if R > 0 else np.nan
    prec = 'alta' if cv < 15 else ('moderada' if cv < 30 else 'baja')
    return dict(indicador=etiqueta, pct=R * 100, se=se * 100, cv=cv, li90=(R - Z90 * se) * 100, ls90=(R + Z90 * se) * 100,
                n_muestra_dominio=int(x.sum()), n_muestra_si=int((y * x).sum()), total_expandido=Y, precision=prec)

def es(df, col, val='1'):
    return (df[col] == val).astype(int).to_numpy()

def tab(filas):
    return pd.DataFrame(filas).round({'pct': 1, 'se': 2, 'cv': 1, 'li90': 1, 'ls90': 1, 'total_expandido': 0})

### 1.1 Validación contra cifras publicadas por el INEGI

Si la función reproduce lo que publicó el INEGI, se puede confiar en ella para el resto del análisis:
- Participación en cuidados de la población de 15 años y más: **32.0%**, que son **31.7 millones** de personas. Se calcula en TCSDEMPO con `P4_69_i` y `FAC_HOG`, siguiendo la guía *Conociendo la base de datos*.
- Hogares que contratan personal de enfermería o de cuidados: **0.8%**. Comunicado 578/23, p. 14.

In [3]:
sde['EDAD_N'] = pd.to_numeric(sde.EDAD, errors='coerce')
cuida = sde[[f'P4_69_{i}' for i in range(1, 7)]].isin(['1', '9']).any(axis=1).astype(int).to_numpy()
p15 = sde.EDAD_N.between(15, 98).astype(int).to_numpy()
v1 = estimar(sde, cuida * p15, p15, 'FAC_HOG', 'Población 15+ que brinda cuidados (publicado: 32.0%, 31.7 M)')

enf_o_cui = ((hog.P2_4_3 == '1') | (hog.P2_4_4 == '1')).astype(int).to_numpy()
v2 = estimar(hog, enf_o_cui, np.ones(len(hog)), 'FAC_HOG', 'Hogares que contratan enfermería o cuidadoras (publicado: 0.8%)')
val = tab([v1, v2]); val

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
0,"Población 15+ que brinda cuidados (publicado: 32.0%, 31.7 M)",32.0,0.51,1.6,31.2,32.8,16590,5758,31652134.0,alta
1,Hogares que contratan enfermería o cuidadoras (publicado: 0.8%),0.7,0.12,18.2,0.5,0.9,6508,45,265054.0,moderada


## 2. ¿Buscan servicios de cuidado? ¿Por qué no?

**Universo principal.** Cuidadoras y cuidadores (TPOB_CUI) de una persona del hogar **con discapacidad** (`FILTRO6_1`) o **de 60 años y más** (`FILTRO6_6`). Es la población objetivo del programa.
- **P6.42:** desde octubre de 2021, ¿buscó directamente una guardería, casa del abuelo o asilo?
- **P6.45:** a quien no buscó, ¿por qué no buscó? Es de respuesta múltiple, así que los porcentajes no suman 100.

In [4]:
RAZONES_NO_BUSCA = {
 'P6_45_01': 'El propio hogar cuida', 'P6_45_02': 'Apoyan otros hogares/familiares/conocidos',
 'P6_45_03': 'Son caros', 'P6_45_04': 'Preocupa el qué dirán', 'P6_45_05': 'DESCONOCE QUE EXISTEN',
 'P6_45_06': 'Incomodidad/desconfianza de que alguien ajeno cuide', 'P6_45_07': 'Lejanía',
 'P6_45_08': 'Maltrato del personal', 'P6_45_09': 'No ha tenido necesidad', 'P6_45_10': 'Otra razón'}

def bloque_busqueda(df, dominio, peso, nombre):
    filas = [estimar(df, es(df, 'P6_42') * dominio, dominio, peso, f'{nombre} | Buscó servicio (P6.42)')]
    no_busco = dominio * es(df, 'P6_42', '2')
    for col, et in RAZONES_NO_BUSCA.items():
        filas.append(estimar(df, es(df, col) * no_busco, no_busco, peso, f'{nombre} | No buscó por: {et}'))
    return filas

dom_obj = ((cui.FILTRO6_1 == '1') | (cui.FILTRO6_6 == '1')).astype(int).to_numpy()
dom_todos = np.ones(len(cui), int)
b_obj = tab(bloque_busqueda(cui, dom_obj, 'FAC_CUI', 'Cuidan discapacidad o 60+'))
b_obj.sort_values('pct', ascending=False)

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
9,Cuidan discapacidad o 60+ | No buscó por: No ha tenido necesidad,49.4,3.30,6.7,44.0,54.8,1019,556,2997224.0,alta
1,Cuidan discapacidad o 60+ | No buscó por: El propio hogar cuida,41.9,3.93,9.4,35.5,48.4,1019,373,2544153.0,alta
2,Cuidan discapacidad o 60+ | No buscó por: Apoyan otros hogares/familiares/co...,6.2,1.14,18.5,4.3,8.0,1019,64,373994.0,moderada
10,Cuidan discapacidad o 60+ | No buscó por: Otra razón,3.2,0.75,23.4,2.0,4.4,1019,35,194990.0,moderada
7,Cuidan discapacidad o 60+ | No buscó por: Lejanía,3.0,0.79,26.5,1.7,4.3,1019,26,179905.0,moderada
3,Cuidan discapacidad o 60+ | No buscó por: Son caros,2.8,0.66,23.5,1.7,3.9,1019,34,170769.0,moderada
0,Cuidan discapacidad o 60+ | Buscó servicio (P6.42),2.5,0.76,30.5,1.2,3.7,1043,24,154125.0,baja
5,Cuidan discapacidad o 60+ | No buscó por: DESCONOCE QUE EXISTEN,1.6,0.62,38.0,0.6,2.6,1019,16,98235.0,baja
4,Cuidan discapacidad o 60+ | No buscó por: Preocupa el qué dirán,1.5,0.63,42.0,0.5,2.5,1019,12,90529.0,baja
6,Cuidan discapacidad o 60+ | No buscó por: Incomodidad/desconfianza de que al...,1.5,0.46,31.9,0.7,2.2,1019,18,88133.0,baja


In [5]:
# Comparación con todas las personas cuidadoras y con la población 15-60 (TPER_ELE)
b_todos = tab(bloque_busqueda(cui, dom_todos, 'FAC_CUI', 'Todas las cuidadoras'))
b_ele = tab(bloque_busqueda(ele, np.ones(len(ele), int), 'FAC_ELE', 'Población 15-60'))
comp = pd.concat([b.assign(razon=b.indicador.map(lambda s: s.split(' | ', 1)[1])).set_index('razon')[['pct', 'cv']]
                  for b in [b_obj, b_todos, b_ele]], axis=1, keys=['Cuidan disc/60+', 'Todas cuidadoras', 'Pob. 15-60'])
comp

Cuidan disc/60+       Todas cuidadoras       Pob. 15-60      
                                                                              pct    cv              pct    cv        pct    cv
razon                                                                                                                          
Buscó servicio (P6.42)                                                        2.5  30.5              3.0  10.6        1.7  12.8
No buscó por: El propio hogar cuida                                          41.9   9.4             26.9   5.1       21.2   5.8
No buscó por: Apoyan otros hogares/familiares/conocidos                       6.2  18.5              4.7  11.6        2.7  14.9
No buscó por: Son caros                                                       2.8  23.5              2.1  10.9        1.5  14.8
No buscó por: Preocupa el qué dirán                                           1.5  42.0              0.5  27.4        0.6  21.7
No buscó por: DESCONOCE QUE EXISTEN                                           1.6  38.0              0.9  26.5        0.9  21.0
No buscó por: Incomodidad/desconfianza de que alguien ajeno cuide             1.5  31.9              1.4  13.9        1.1  18.3
No buscó por: Lejanía                                                         3.0  26.5              1.9  12.7        1.2  14.7
No buscó por: Maltrato del personal                                           1.3  35.6              0.9  21.2        0.5  36.1
No buscó por: No ha tenido necesidad                                         49.4   6.7             67.3   2.0       75.7   1.7
No buscó por: Otra razón                                                      3.2  23.4              1.5  17.2        1.0  20.4

## 3. Quienes sí buscaron: ¿encontraron lo que necesitaban? (P6.43–P6.44)

Muy pocos buscaron, así que esta parte tiene muestra chica. Solo se leen las estimaciones de precisión alta o moderada.

In [6]:
RAZONES_NO_CUMPLE = {f'P6_44_{i:02d}': e for i, e in enumerate(
    ['Costo elevado', 'Personal no capacitado', 'Lejanía', 'Instalaciones inadecuadas', 'Sin traslados',
     'LISTA DE ESPERA (cupo insuficiente)', 'Sin flexibilidad de horario', 'Sin alimentos/dieta',
     'No aceptan movilidad limitada', 'No dan cuidados especializados', 'Otra'], start=1)}
busco = es(cui, 'P6_42')
filas = [estimar(cui, es(cui, 'P6_43') * busco, busco, 'FAC_CUI', 'Todas cuidadoras que buscaron | Cumplió lo que buscaba')]
no_cumplio = busco * es(cui, 'P6_43', '2')
for col, et in RAZONES_NO_CUMPLE.items():
    filas.append(estimar(cui, es(cui, col) * no_cumplio, no_cumplio, 'FAC_CUI', f'No cumplió | {et}'))
b_busco = tab(filas); b_busco.sort_values('pct', ascending=False)

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
0,Todas cuidadoras que buscaron | Cumplió lo que buscaba,69.3,4.95,7.1,61.2,77.4,185,139,661625.0,alta
1,No cumplió | Costo elevado,30.9,10.12,32.8,14.2,47.5,46,11,90536.0,baja
2,No cumplió | Personal no capacitado,20.8,7.70,37.1,8.1,33.4,46,9,60900.0,baja
3,No cumplió | Lejanía,19.6,6.77,34.6,8.4,30.7,46,9,57432.0,baja
7,No cumplió | Sin flexibilidad de horario,15.5,6.77,43.8,4.3,26.6,46,7,45349.0,baja
11,No cumplió | Otra,14.2,5.83,41.0,4.6,23.8,46,7,41630.0,baja
10,No cumplió | No dan cuidados especializados,13.7,5.49,40.1,4.7,22.7,46,7,40147.0,baja
4,No cumplió | Instalaciones inadecuadas,11.0,4.40,40.0,3.8,18.2,46,7,32270.0,baja
6,No cumplió | LISTA DE ESPERA (cupo insuficiente),7.8,4.33,55.6,0.7,14.9,46,4,22832.0,baja
5,No cumplió | Sin traslados,5.6,4.02,72.2,-1.0,12.2,46,2,16327.0,baja


## 4. Si tuviera que contratar cuidado, ¿dónde lo preferiría? (P6.46)

In [7]:
OPC46 = {'1': 'En su casa', '2': 'Centro/institución pública', '3': 'Centro/institución privada',
         '4': 'Casa de familiar/amistad/conocido', '5': 'No contrataría'}
pref = []
for nombre, df, dom, peso in [('Cuidan disc/60+', cui, dom_obj, 'FAC_CUI'), ('Pob. 15-60', ele, np.ones(len(ele), int), 'FAC_ELE')]:
    for k, et in OPC46.items():
        pref.append(estimar(df, es(df, 'P6_46', k) * dom, dom, peso, f'{nombre} | {et}'))
b_pref = tab(pref); b_pref

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
0,Cuidan disc/60+ | En su casa,58.1,2.74,4.7,53.6,62.7,1043,657,3618967.0,alta
1,Cuidan disc/60+ | Centro/institución pública,3.5,0.91,25.9,2.0,5.0,1043,30,217824.0,moderada
2,Cuidan disc/60+ | Centro/institución privada,0.8,0.37,44.5,0.2,1.4,1043,10,51870.0,baja
3,Cuidan disc/60+ | Casa de familiar/amistad/conocido,1.6,0.49,30.1,0.8,2.4,1043,17,101669.0,baja
4,Cuidan disc/60+ | No contrataría,35.9,2.63,7.3,31.6,40.2,1043,329,2233354.0,alta
5,Pob. 15-60 | En su casa,63.8,1.27,2.0,61.7,65.9,5579,3740,51221511.0,alta
6,Pob. 15-60 | Centro/institución pública,2.7,0.31,11.4,2.2,3.2,5579,167,2152189.0,alta
7,Pob. 15-60 | Centro/institución privada,2.4,0.27,11.0,2.0,2.9,5579,137,1949311.0,alta
8,Pob. 15-60 | Casa de familiar/amistad/conocido,2.8,0.30,10.7,2.3,3.3,5579,174,2255661.0,alta
9,Pob. 15-60 | No contrataría,28.2,1.15,4.1,26.3,30.1,5579,1361,22658389.0,alta


## 5. Lado "Puedo cuidar": ¿le gustaría ser cuidador(a) profesional si le pagaran? (P6.47–P6.48)

**Advertencia.** Es una intención declarada ante una pregunta hipotética. No mide la oferta real de cuidadoras, ni con qué capacitación ni a qué precio.

In [8]:
sup = []
for nombre, df, dom, peso in [('Pob. 15-60', ele, np.ones(len(ele), int), 'FAC_ELE'), ('Todas cuidadoras', cui, dom_todos, 'FAC_CUI')]:
    sup.append(estimar(df, es(df, 'P6_47') * dom, dom, peso, f'{nombre} | Le gustaría ser cuidador(a) pagada(o)'))
    si47 = dom * es(df, 'P6_47')
    sup.append(estimar(df, es(df, 'P6_48') * si47, si47, peso, f'{nombre} | ...y se capacitaría (entre quienes sí)'))
ele['MUJER'] = (ele.SEXO == '2').astype(int)
for s, et in [(1, 'Mujeres'), (0, 'Hombres')]:
    d = (ele.MUJER == s).astype(int).to_numpy()
    sup.append(estimar(ele, es(ele, 'P6_47') * d, d, 'FAC_ELE', f'Pob. 15-60 {et} | Le gustaría ser cuidador(a) pagada(o)'))
b_sup = tab(sup); b_sup

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
0,Pob. 15-60 | Le gustaría ser cuidador(a) pagada(o),40.2,1.09,2.7,38.4,42.0,5579,2467,32236744.0,alta
1,Pob. 15-60 | ...y se capacitaría (entre quienes sí),95.0,0.55,0.6,94.1,95.9,2467,2348,30612377.0,alta
2,Todas cuidadoras | Le gustaría ser cuidador(a) pagada(o),41.7,1.07,2.6,40.0,43.5,5677,2572,13202047.0,alta
3,Todas cuidadoras | ...y se capacitaría (entre quienes sí),96.3,0.39,0.4,95.7,97.0,2572,2478,12715647.0,alta
4,Pob. 15-60 Mujeres | Le gustaría ser cuidador(a) pagada(o),45.0,1.41,3.1,42.7,47.3,3017,1499,19129404.0,alta
5,Pob. 15-60 Hombres | Le gustaría ser cuidador(a) pagada(o),34.7,1.43,4.1,32.4,37.1,2562,968,13107340.0,alta


## 6. Contratación y precio del cuidado pagado (THOGAR, P2.4–P2.8, P3A.4)

**Pregunta.** En los hogares con una persona con discapacidad (`HN_CDISC`) o de 60 años y más (`HN_C60MA`), ¿cuántos contratan cuidado pagado? ¿Y cuánto cuesta ese cuidado frente al ingreso del hogar?

**Advertencia de muestra.** Solo 49 hogares de toda la muestra nacional contratan enfermería o cuidadoras. El precio se reporta como **descriptivo de muestra**, no como estimación poblacional.

In [9]:
dom_disc = es(hog, 'HN_CDISC'); dom_60 = es(hog, 'HN_C60MA'); dom_cual = ((dom_disc + dom_60) > 0).astype(int)
con = []
for d, et in [(dom_disc, 'Hogares con persona con discapacidad'), (dom_60, 'Hogares con persona 60+'), (dom_cual, 'Hogares con discapacidad o 60+')]:
    con.append(estimar(hog, enf_o_cui * d, d, 'FAC_HOG', f'{et} | Contrata enfermería o cuidadora'))
    con.append(estimar(hog, es(hog, 'P2_4_1') * d, d, 'FAC_HOG', f'{et} | Contrata trabajo doméstico (entrada por salida)'))
b_con = tab(con); b_con

,indicador,pct,se,cv,li90,ls90,n_muestra_dominio,n_muestra_si,total_expandido,precision
0,Hogares con persona con discapacidad | Contrata enfermería o cuidadora,2.9,0.72,24.5,1.8,4.1,877,24,145797.0,moderada
1,Hogares con persona con discapacidad | Contrata trabajo doméstico (entrada p...,7.6,1.28,16.7,5.5,9.7,877,68,377681.0,moderada
2,Hogares con persona 60+ | Contrata enfermería o cuidadora,0.6,0.22,35.9,0.2,1.0,1983,13,74361.0,baja
3,Hogares con persona 60+ | Contrata trabajo doméstico (entrada por salida),8.0,0.91,11.4,6.5,9.5,1983,153,991110.0,alta
4,Hogares con discapacidad o 60+ | Contrata enfermería o cuidadora,1.0,0.24,23.6,0.6,1.4,2554,26,159010.0,moderada
5,Hogares con discapacidad o 60+ | Contrata trabajo doméstico (entrada por sal...,7.8,0.81,10.3,6.5,9.2,2554,195,1212877.0,alta


In [10]:
def num(s, invalidos=('99999',)):
    return pd.to_numeric(s.where(~s.isin(invalidos)), errors='coerce')

pagos = pd.concat([num(hog.loc[hog.P2_4_3 == '1', 'P2_8_3']).rename('pago').to_frame().assign(tipo='enfermería'),
                   num(hog.loc[hog.P2_4_4 == '1', 'P2_8_4']).rename('pago').to_frame().assign(tipo='cuidadora')])
print('Pago semanal ($) — hogares de la muestra que contratan (sin ponderar):')
print(pagos.groupby('tipo').pago.describe(percentiles=[.25, .5, .75]).round(0))

hog['ING'] = num(hog.P3A_4, ('99999',))
ing = hog.loc[dom_cual == 1, 'ING'].dropna()
w = pd.to_numeric(hog.loc[ing.index, 'FAC_HOG'])
def wq(v, w, q):
    o = np.argsort(v.values); c = np.cumsum(w.values[o]) / w.sum(); return v.values[o][np.searchsorted(c, q)]
med_ing = wq(ing, w, .5); p25_ing = wq(ing, w, .25)
med_pago_mes = pagos.pago.median() * 4.33
print(f'\nHogares con discapacidad o 60+ que reportan ingreso: n={len(ing)} de {dom_cual.sum()}')
print(f'Ingreso mensual ponderado: p25=${p25_ing:,.0f}  mediana=${med_ing:,.0f}')
print(f'Pago mensual mediano de cuidado contratado (muestra) ≈ ${med_pago_mes:,.0f}  -> {med_pago_mes / med_ing * 100:.0f}% del ingreso mediano, {med_pago_mes / p25_ing * 100:.0f}% del p25')
precio = pd.DataFrame([
    dict(indicador='Pago semanal mediano, cuidadora contratada (muestra, sin ponderar)', valor=pagos.loc[pagos.tipo=='cuidadora','pago'].median(), n=int(pagos.loc[pagos.tipo=='cuidadora','pago'].notna().sum())),
    dict(indicador='Pago semanal mediano, enfermería contratada (muestra, sin ponderar)', valor=pagos.loc[pagos.tipo=='enfermería','pago'].median(), n=int(pagos.loc[pagos.tipo=='enfermería','pago'].notna().sum())),
    dict(indicador='Pago semanal mediano, cuidadora o enfermería (muestra, sin ponderar)', valor=pagos.pago.median(), n=int(pagos.pago.notna().sum())),
    dict(indicador='Pago mensual equivalente (mediana semanal x 4.33)', valor=round(med_pago_mes), n=int(pagos.pago.notna().sum())),
    dict(indicador='Ingreso mensual mediano ponderado, hogares con discapacidad o 60+', valor=med_ing, n=len(ing)),
    dict(indicador='Ingreso mensual p25 ponderado, hogares con discapacidad o 60+', valor=p25_ing, n=len(ing)),
    dict(indicador='Pago mensual como % del ingreso mediano', valor=round(med_pago_mes/med_ing*100, 1), n=None),
    dict(indicador='Pago mensual como % del ingreso p25', valor=round(med_pago_mes/p25_ing*100, 1), n=None),
])
precio.to_csv(OUT / '03_precio_vs_ingreso_muestra.csv', index=False, encoding='utf-8-sig')
precio

Pago semanal ($) — hogares de la muestra que contratan (sin ponderar):
            count    mean     std    min    25%    50%     75%      max
tipo                                                                   
cuidadora    29.0  1751.0  2830.0  180.0  450.0  800.0  1500.0  12000.0
enfermería   16.0  2451.0  3506.0  250.0  375.0  700.0  3000.0  10000.0

Hogares con discapacidad o 60+ que reportan ingreso: n=2054 de 2554
Ingreso mensual ponderado: p25=$3,900  mediana=$6,500
Pago mensual mediano de cuidado contratado (muestra) ≈ $3,464  -> 53% del ingreso mediano, 89% del p25


,indicador,valor,n
0,"Pago semanal mediano, cuidadora contratada (muestra, sin ponderar)",800.0,29.0
1,"Pago semanal mediano, enfermería contratada (muestra, sin ponderar)",700.0,16.0
2,"Pago semanal mediano, cuidadora o enfermería (muestra, sin ponderar)",800.0,45.0
3,Pago mensual equivalente (mediana semanal x 4.33),3464.0,45.0
4,"Ingreso mensual mediano ponderado, hogares con discapacidad o 60+",6500.0,2054.0
5,"Ingreso mensual p25 ponderado, hogares con discapacidad o 60+",3900.0,2054.0
6,Pago mensual como % del ingreso mediano,53.3,NaN
7,Pago mensual como % del ingreso p25,88.8,NaN


In [11]:
todo = pd.concat([val.assign(bloque='0_validacion'), b_obj.assign(bloque='2_busqueda_cuidan_disc60'),
                  b_todos.assign(bloque='2_busqueda_todas_cuidadoras'), b_ele.assign(bloque='2_busqueda_pob15_60'),
                  b_busco.assign(bloque='3_quienes_buscaron'), b_pref.assign(bloque='4_preferencia_lugar'),
                  b_sup.assign(bloque='5_oferta_declarada'), b_con.assign(bloque='6_contratacion')])
todo.to_csv(OUT / '03_enasic_barreras_estimaciones.csv', index=False, encoding='utf-8-sig')
print(len(todo), 'estimaciones guardadas;', (todo.precision == 'baja').sum(), 'con precisión baja')

69 estimaciones guardadas; 20 con precisión baja


## 7. Lectura

Todo lo que sigue es **nacional**. Los números se leen de `resultados/03_enasic_barreras_estimaciones.csv`.

**0. La función está validada.** Reproduce la participación en cuidados publicada por el INEGI: 32.0% y 31.7 millones de personas.
En contratación de enfermería o cuidadoras da 0.7%. El INEGI publicó 0.8%: el valor publicado cae dentro del intervalo de confianza al 90% (0.5–0.9), y el total, 0.27 millones, redondea a los "0.3 millones" del comunicado.

**1. Casi nadie busca servicios de cuidado, y no es por desconocimiento.**
- De quienes cuidan a una persona con discapacidad o de 60+, muy pocos buscaron una guardería, casa del abuelo o asilo. La estimación de este grupo tiene precisión baja; entre todas las personas cuidadoras, el 3.0% buscó (precisión alta).
- De quienes no buscaron:
  - 49% dice que no tuvo necesidad.
  - 42% dice que el propio hogar cuida.
  - "Desconoce que existen" es 0.9% en todas las cuidadoras (precisión moderada). En la población objetivo la estimación tiene precisión baja y no se usa.
  - "Son caros" es alrededor de 2–3% y "desconfianza" alrededor de 1.5%.
- **La hipótesis de que la barrera es de información no se sostiene para servicios institucionales.** La barrera dominante es la norma de que la familia cuida o la necesidad no percibida.

**2. De quienes sí buscaron, 69% encontró lo que necesitaba** (precisión alta). Entre quienes no, los motivos (costo, personal, lejanía) tienen precisión baja y no se leen como hallazgo.

**3. La preferencia es el cuidado en casa.** Si tuvieran que contratar, 58% de quienes cuidan a una persona con discapacidad o de 60+ lo preferiría en su casa, 36% no contrataría nada y menos del 5% elegiría una institución.

**4. Hay oferta declarada.** 40% de la población de 15 a 60 años (45% de las mujeres) diría que sí a ser cuidador(a) si le pagaran, y 95% de ellas y ellos se capacitaría. Es intención ante una pregunta hipotética, no oferta real.

**5. Se contrata muy poco y el precio pesa.**
- Solo alrededor de 1% de los hogares con una persona con discapacidad o de 60+ contrata enfermería o cuidadora (precisión moderada). En cambio, 7.8% contrata trabajo doméstico.
- En la muestra, el pago semanal mediano por cuidado contratado es de unos $800. Son unos $3,500 al mes: 53% del ingreso mensual mediano de los hogares con necesidad ($6,500) y 89% del percentil 25.
- Este cálculo es descriptivo: son 45 hogares, sin ponderar, y el pago depende de las horas contratadas.

**Qué implica para la app (probable, nacional):**
- **No hay evidencia de que conectar oferta con demanda mueva la contratación.** La información no es la barrera: la gente sí sabe que existen los servicios.
- Lo que se observa es:
  - preferencia por cuidar en casa;
  - oferta declarada abundante;
  - precio alto frente al ingreso;
  - una norma familiar fuerte.
- Un matching sin un componente de precio (subsidio, cuidado comunitario o cooperativo) atacaría una barrera que los datos no muestran.

**Límites.** P6.45 pregunta por *lugares* (guardería, casa del abuelo, asilo). **No pregunta por contratar a una persona que cuide en casa.** La barrera específica para contratar cuidado domiciliario sigue sin medirse directamente.
Nada de esto es para la CDMX. Solo hay una edición, así que no hay tendencias.